In [2]:
import pandas as pd

df_games = pd.read_csv('../data/interim/historical_matches.csv')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
print(df_games.shape)
df_games.head(2)

(49215, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False


In [4]:
df_games['tournament'].value_counts().head()

tournament
Friendly                                18252
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                            964
Name: count, dtype: int64

In [5]:
# Exclude non-FIFA tournaments / Matches

df_games = df_games[~df_games['tournament'].str.startswith('CONIFA')]
df_games = df_games[~df_games['tournament'].str.startswith('ConIFA')]

print(df_games.shape)

(49048, 9)


# Compute Elo ratings for every international team over history

In [6]:
import pandas as pd
from collections import defaultdict

# ============================================================
# Compute Elo ratings for every international team over history
# ============================================================

# ---------- 1. K-factor tiers by tournament importance ----------
TIER_1_WORLD_CUP = {'FIFA World Cup'}

TIER_2_CONTINENTAL = {
    'UEFA Euro', 'Copa América', 'African Cup of Nations', 'AFC Asian Cup',
    'Gold Cup', 'CONCACAF Championship', 'Oceania Nations Cup',
    'Confederations Cup',
}

TIER_3_QUALIFIERS_NATIONS = {
    'FIFA World Cup qualification', 'UEFA Euro qualification',
    'African Cup of Nations qualification', 'AFC Asian Cup qualification',
    'Gold Cup qualification', 'CONCACAF Championship qualification',
    'Copa América qualification', 'Oceania Nations Cup qualification',
    'UEFA Nations League', 'CONCACAF Nations League',
    'CONCACAF Nations League qualification',
}

TIER_4_REGIONAL = {
    # Africa
    'CECAFA Cup', 'COSAFA Cup', 'COSAFA Cup qualification', 'WAFF Championship',
    'Amílcar Cabral Cup', 'All-African Games', 'UDEAC Cup', 'UNIFFAC Cup',
    'West African Cup', 'Nile Basin Tournament', 'African Friendship Games',
    # Asia / Oceania
    'Gulf Cup', 'Arab Cup', 'Arab Cup qualification', 'SAFF Cup',
    'AFF Championship', 'AFF Championship qualification', 'EAFF Championship',
    'EAFF Championship qualification', 'ASEAN Championship',
    'ASEAN Championship qualification', 'AFC Challenge Cup',
    'AFC Challenge Cup qualification', 'Asian Games', 'CAFA Nations Cup',
    'Southeast Asian Games', 'South Asian Games', 'Dynasty Cup',
    'Pacific Games', 'South Pacific Games', 'Melanesia Cup',
    'Indian Ocean Island Games', 'Afro-Asian Games',
    # Europe
    'British Home Championship', 'Nordic Championship', 'Baltic Cup',
    'Balkan Cup', 'Central European International Cup',
    # Americas
    'CFU Caribbean Cup', 'CFU Caribbean Cup qualification', 'UNCAF Cup',
    'Central American and Caribbean Games', 'Pan American Championship',
    'CCCF Championship', 'Bolivarian Games', 'NAFC Championship',
    # Multi-sport
    'Olympic Games',
}

TIER_5_FRIENDLY = {'Friendly', 'FIFA Series', 'CONCACAF Series'}


def get_k_factor(tournament):
    """K controls how reactive Elo is to a match. Higher = bigger swings."""
    if tournament in TIER_1_WORLD_CUP:          return 60
    if tournament in TIER_2_CONTINENTAL:        return 50
    if tournament in TIER_3_QUALIFIERS_NATIONS: return 40
    if tournament in TIER_4_REGIONAL:           return 30
    if tournament in TIER_5_FRIENDLY:           return 20
    return 15   # minor/exhibition tournaments, anniversary cups, etc.


# ---------- 2. Expected score formula ----------
def expected_score(home_elo, away_elo, home_advantage=100):
    # home_advantage = 100 → ~64% win rate for evenly matched home teams.
    # Set to 0 for neutral-venue matches (the formula then becomes symmetric).
    diff = (home_elo + home_advantage) - away_elo
    return 1 / (1 + 10 ** (-diff / 400))


# ---------- 3. Load and prepare ----------
df = df_games.copy()
df = df.dropna(subset=['home_score', 'away_score'])

# Drop CONIFA tournaments — these only involve non-FIFA teams
df = df[~df['tournament'].str.contains('CONIFA', case=False, na=False)]

df = df.sort_values('date').reset_index(drop=True)
print(f"Computing Elo over {len(df):,} historical matches")

# ---------- 4. Iterate through history ----------
INITIAL_ELO = 1500
current_elo = defaultdict(lambda: INITIAL_ELO)
elo_history = []

for row in df.itertuples(index=False):
    home, away = row.home_team, row.away_team
    home_elo, away_elo = current_elo[home], current_elo[away]

    # Neutral venue cancels home advantage
    h_adv = 0 if row.neutral else 100
    exp_home = expected_score(home_elo, away_elo, h_adv)
    exp_away = 1 - exp_home

    # Actual result
    if row.home_score > row.away_score:
        actual_home, actual_away = 1.0, 0.0
    elif row.home_score < row.away_score:
        actual_home, actual_away = 0.0, 1.0
    else:
        actual_home, actual_away = 0.5, 0.5

    # Goal-difference multiplier — bigger wins move Elo more
    goal_diff = abs(row.home_score - row.away_score)
    if goal_diff <= 1:
        g = 1.0
    elif goal_diff == 2:
        g = 1.5
    else:
        g = (11 + goal_diff) / 8

    K = get_k_factor(row.tournament)

    # Update both teams (zero-sum: total Elo across the two teams is preserved)
    current_elo[home] = home_elo + K * g * (actual_home - exp_home)
    current_elo[away] = away_elo + K * g * (actual_away - exp_away)

    elo_history.append((row.date, home, current_elo[home]))
    elo_history.append((row.date, away, current_elo[away]))

# ---------- 5. Build the lookup table ----------
df_elo = pd.DataFrame(elo_history, columns=['date', 'team', 'elo_after'])
df_elo['date'] = pd.to_datetime(df_elo['date'])
print(f"df_elo shape: {df_elo.shape}")

# ---------- 6. Sanity checks ----------
print("\n--- Top 15 current Elo ---")
current_ranking = (
    df_elo.sort_values('date')
          .groupby('team')
          .tail(1)
          .sort_values('elo_after', ascending=False)
          .head(15)
)
print(current_ranking.to_string(index=False))

print("\n--- Bottom 15 current Elo (sanity: should be tiny non-FIFA or minnow teams) ---")
print(
    df_elo.sort_values('date')
          .groupby('team')
          .tail(1)
          .sort_values('elo_after')
          .head(15)
          .to_string(index=False)
)

print(f"\nTotal unique teams with Elo: {df_elo['team'].nunique()}")

Computing Elo over 49,048 historical matches
df_elo shape: (98096, 3)

--- Top 15 current Elo ---
      date        team   elo_after
2026-03-31       Spain 2220.668648
2026-03-31   Argentina 2180.246575
2026-03-29      France 2137.182107
2026-03-31     England 2078.893971
2026-03-31      Brazil 2062.818926
2026-03-29    Colombia 2051.621962
2026-03-31    Portugal 2034.648091
2026-03-31     Ecuador 2018.272845
2026-03-31 Netherlands 2016.511156
2026-03-30     Germany 1988.982584
2026-03-31     Croatia 1982.130309
2026-03-31     Morocco 1981.410473
2026-03-31       Japan 1978.864707
2026-03-31     Uruguay 1972.555119
2026-03-31      Norway 1961.244369

--- Bottom 15 current Elo (sanity: should be tiny non-FIFA or minnow teams) ---
      date                         team   elo_after
2026-03-29                        Macau  861.915259
2025-11-18                       Bhutan  876.100843
2026-03-29                     Anguilla  899.114241
2025-11-18                       Brunei  907.510912
2

### -> FIFA Ranking AND highly correlated but not identical — exactly what you want. 

##  neutral-flag distribution

In [7]:
print("Neutral venue distribution:")
print(df_games['neutral'].value_counts(normalize=True))
print("\nNeutral by tournament (top 10):")
print(df_games.groupby('tournament')['neutral']
              .mean()
              .sort_values(ascending=False)
              .head())

Neutral venue distribution:
neutral
False    0.738195
True     0.261805
Name: proportion, dtype: float64

Neutral by tournament (top 10):
tournament
FIFA 75th Anniversary Cup    1.0
Hungary Heritage Cup         1.0
Tynwald Hill Tournament      1.0
Al Ain International Cup     1.0
Coupe de l'Outre-Mer         1.0
Name: neutral, dtype: float64


## Quick checks

In [8]:
# Spot check: Morocco should have spiked after WC2022
morocco = df_elo[df_elo['team'] == 'Morocco'].sort_values('date')
print(morocco.tail(20))   # look for jump around Dec 2022

# Spot check: a team that played few matches should be near 1500
print(df_elo[df_elo['team'] == 'Anguilla'].tail(5))

            date     team    elo_after
96849 2025-09-08  Morocco  1923.634935
97018 2025-10-09  Morocco  1924.916547
97280 2025-10-14  Morocco  1925.643214
97396 2025-11-14  Morocco  1926.680570
97564 2025-11-18  Morocco  1929.022437
97636 2025-12-02  Morocco  1932.438844
97653 2025-12-05  Morocco  1920.795340
97668 2025-12-08  Morocco  1927.869667
97680 2025-12-11  Morocco  1932.011323
97690 2025-12-15  Morocco  1941.613564
97693 2025-12-18  Morocco  1950.456477
97694 2025-12-21  Morocco  1953.029306
97724 2025-12-26  Morocco  1934.337446
97746 2025-12-29  Morocco  1939.592641
97772 2026-01-04  Morocco  1941.321060
97784 2026-01-09  Morocco  1951.783809
97792 2026-01-14  Morocco  1938.775828
97798 2026-01-18  Morocco  1971.653463
97920 2026-03-27  Morocco  1972.941440
98090 2026-03-31  Morocco  1981.410473
            date      team   elo_after
96330 2025-06-07  Anguilla  887.863882
97297 2025-11-12  Anguilla  902.034635
97425 2025-11-15  Anguilla  894.689058
97841 2026-03-26  Anguill

In [9]:
df_games.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [10]:
df_elo.head()

,date,team,elo_after
0,1872-11-30,Scotland,1497.198700
1,1872-11-30,England,1502.801300
2,1873-03-08,England,1513.377469
3,1873-03-08,Scotland,1486.622531
4,1874-03-07,Scotland,1494.545054


# Elo impact on fixtures (with Leakage but removed later)

In [11]:
# For each historical match, find each team's Elo state around that match.
# We'll grab BOTH the pre-match Elo (legitimate feature) and the post-match Elo
# (which we'll use to demonstrate target leakage, then drop before training).

df_games['date'] = pd.to_datetime(df_games['date'])
df_elo['date']   = pd.to_datetime(df_elo['date'])

df_elo_sorted   = df_elo.sort_values('date').reset_index(drop=True)
df_games_sorted = df_games.sort_values('date').reset_index(drop=True)

# ----- HOME: pre-match Elo (lookup strictly BEFORE the match date) -----
df_temp = pd.merge_asof(
    df_games_sorted,
    df_elo_sorted.rename(columns={'team': 'home_team', 'elo_after': 'home_elo_pre'}),
    on='date',
    by='home_team',
    direction='backward',
    allow_exact_matches=False,   # exclude same-day entries → no leakage
)

# ----- HOME: post-match Elo (lookup ON the match date — this is the leak) -----
df_temp = pd.merge_asof(
    df_temp.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'home_team', 'elo_after': 'home_elo_after'}),
    on='date',
    by='home_team',
    direction='backward',
    allow_exact_matches=True,    # INCLUDE same-day → grabs the post-match update
)

# ----- AWAY: pre-match Elo -----
df_temp = pd.merge_asof(
    df_temp.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'away_team', 'elo_after': 'away_elo_pre'}),
    on='date',
    by='away_team',
    direction='backward',
    allow_exact_matches=False,
)

# ----- AWAY: post-match Elo (the leak again) -----
df_match_features = pd.merge_asof(
    df_temp.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'away_team', 'elo_after': 'away_elo_after'}),
    on='date',
    by='away_team',
    direction='backward',
    allow_exact_matches=True,
)

df_match_features['elo_diff'] = (
    df_match_features['home_elo_pre'] - df_match_features['away_elo_pre']
)

# Inspect: pre vs after side by side
df_match_features[[
    'date', 'home_team', 'away_team', 'home_score', 'away_score',
    'home_elo_pre', 'home_elo_after', 'away_elo_pre', 'away_elo_after'
]].tail(10)

,date,home_team,away_team,home_score,away_score,home_elo_pre,home_elo_after,away_elo_pre,away_elo_after
49038,2026-03-31,Iraq,Bolivia,2.0,1.0,1710.777393,1733.899556,1765.462067,1742.339904
49039,2026-03-31,Bosnia and Herzegovina,Italy,1.0,1.0,1632.960817,1642.884095,1922.036782,1912.113504
49040,2026-03-31,Sweden,Poland,3.0,2.0,1754.883707,1771.742389,1799.857926,1782.999243
49041,2026-03-31,Kosovo,Turkey,0.0,1.0,1789.903507,1772.593654,1936.921078,1954.230931
49042,2026-03-31,Czech Republic,Denmark,2.0,2.0,1773.840408,1776.737286,1924.520757,1921.623879
49043,2026-03-31,Peru,Honduras,2.0,2.0,1763.677644,1762.044919,1706.438665,1708.071391
49044,2026-03-31,Spain,Egypt,0.0,0.0,2229.756779,2220.668648,1801.424265,1810.512397
49045,2026-03-31,Wales,Northern Ireland,1.0,1.0,1757.649465,1752.167437,1643.695753,1649.177781
49046,2026-03-31,Australia,Curaçao,5.0,1.0,1883.419176,1887.564123,1621.162116,1617.017169
49047,2026-03-31,Kazakhstan,Comoros,1.0,0.0,1489.055719,1496.246250,1488.747627,1481.557097


# Elo on fixtures

In [12]:
# For each historical match, find each team's Elo *just before* that match.
# (You want pre-match Elo as the feature, not post-match — post-match would leak the result.)

df_games['date'] = pd.to_datetime(df_games['date'])
df_elo['date'] = pd.to_datetime(df_elo['date'])

df_elo_sorted = df_elo.sort_values('date').reset_index(drop=True)

# Use merge_asof for "find the most recent Elo entry before this match date"
df_games_sorted = df_games.sort_values('date').reset_index(drop=True)

# Home team Elo at time of match
df_with_home_elo = pd.merge_asof(
    df_games_sorted,
    df_elo_sorted.rename(columns={'team': 'home_team', 'elo_after': 'home_elo_pre'}),
    on='date',
    by='home_team',
    direction='backward',
    allow_exact_matches=False  # don't pick up the *current* match's post-Elo
)

# Same for away team
df_match_features = pd.merge_asof(
    df_with_home_elo.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'away_team', 'elo_after': 'away_elo_pre'}),
    on='date',
    by='away_team',
    direction='backward',
    allow_exact_matches=False
)

df_match_features['elo_diff'] = df_match_features['home_elo_pre'] - df_match_features['away_elo_pre']

In [13]:
print(df_match_features.shape)
df_match_features.tail(2)

(49048, 12)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff
49046,2026-03-31,Australia,Curaçao,5.0,1.0,FIFA Series,Melbourne,Australia,False,1883.419176,1621.162116,262.257060
49047,2026-03-31,Kazakhstan,Comoros,1.0,0.0,FIFA Series,Astana,Kazakhstan,False,1489.055719,1488.747627,0.308092


In [14]:
assert 'home_elo_after' not in df_match_features.columns
assert 'away_elo_after' not in df_match_features.columns
assert 'home_elo_change' not in df_match_features.columns
print("✓ No leaky columns")

✓ No leaky columns


In [16]:
df_match_features.to_csv('../data/processed/df_match_features.csv', index=False)
print(f"Saved: {df_match_features.shape}")

Saved: (49048, 12)
